# Sonaveil — fine-tune smoke test (Google Colab)

Runs the **exact same** `training/finetune.py` from the repo on Colab's free GPU.
Nothing is reimplemented here — these cells just set up the environment, point at
your project on Google Drive, and call the script.

**Before running:** set Runtime → Change runtime type → **GPU (T4)**, and make sure
your project folder (with `training/` and `data/mixtures/`) is on your Drive.

Goal of a smoke test: confirm it loads pretrained weights, the loss goes **down**,
and a checkpoint saves. Quality is *not* the point — it will be undertrained.

### 1. Mount Google Drive
Authorize when prompted. Your files appear under `/content/drive/MyDrive/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Point at the project on Drive
Edit `PROJECT` to wherever you uploaded the repo (the folder containing
`training/` and `data/mixtures/`). The cell verifies the data is there.

In [ ]:
import os

PROJECT = '/content/drive/MyDrive/sonaveil'  # <-- edit if yours differs
os.chdir(PROJECT)

n_mix = len([d for d in os.listdir('data/mixtures')
             if d.startswith('track_')]) if os.path.isdir('data/mixtures') else 0
print('project :', PROJECT)
print('mixtures:', n_mix)
assert n_mix > 0, 'No mixtures found — upload data/mixtures/ to Drive (run make_mixtures.py locally first).'

### 3. Install demucs + confirm GPU
Colab already ships a CUDA build of torch, so this won't reinstall it — it just
adds demucs and its helpers. The assert fails loudly if you forgot to pick a GPU runtime.

In [ ]:
!pip install -q demucs

import torch
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu    :', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU.'

### 4. Run the smoke test (the real `finetune.py`)
Checkpoints are written to `ckpt/` **on Drive**, so if Colab disconnects you can
re-run this cell and it resumes from the last epoch. First run also downloads the
pretrained `htdemucs_6s` weights (~once).

Watch for: `train L1` and `val L1` decreasing across the 2 epochs.

In [ ]:
!python training/finetune.py \
    --epochs 2 \
    --segment 5 \
    --batch-size 4 \
    --checkpoint-dir ckpt

### 5. (Optional) Export to a loadable model
Only needed if you want to actually *hear* the (undertrained) separation in app.py.
Writes `models/sonaveil_v1/` on Drive. Not required to validate that training works.

In [ ]:
!python training/export.py --checkpoint ckpt/best.pt --name sonaveil_v1

---
**If the loss went down and a checkpoint saved — the smoke test passed.** The same
`finetune.py` then runs on RunPod for the real training: more data, bigger
`--batch-size`/`--segment`, more `--epochs`.